# Chapter 10: Completeness, Lehmann–Scheffé and the UMVUE

The computational applications of this chapter of *Theory of Statistics*. The code is generated automatically from the book's `code/ch10/` files; see the book for the explanations and proofs.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibrahimguney/statistical-theory/blob/main/notebooks/ch_10.ipynb)

$$
\newcommand{\R}{\mathbb{R}}\newcommand{\N}{\mathbb{N}}\newcommand{\E}{\mathbb{E}}
\newcommand{\P}{\mathbb{P}}\newcommand{\F}{\mathcal{F}}\newcommand{\I}{\mathbb{I}}
\newcommand{\Var}{\operatorname{Var}}\newcommand{\Cov}{\operatorname{Cov}}
\newcommand{\Corr}{\operatorname{Corr}}\newcommand{\MSE}{\operatorname{MSE}}
\newcommand{\Bias}{\operatorname{Bias}}\newcommand{\Normal}{\mathcal{N}}
\newcommand{\iid}{\overset{\text{i.i.d.}}{\sim}}\newcommand{\pto}{\xrightarrow{P}}
\newcommand{\dto}{\xrightarrow{d}}\newcommand{\X}{\mathbf{X}}\newcommand{\x}{\mathbf{x}}
$$

In [1]:
# Setup: numpy, scipy and matplotlib are preinstalled on Colab. Locally:
#   pip install -r code/requirements.txt
# Figure files (PDF) saved by the code are written to the _figures/ folder; the figures are
# also shown inside the notebook.
import os
os.makedirs("_figures", exist_ok=True)
os.chdir("_figures")
%matplotlib inline

## The German Tank Problem

*Source file:* `code/ch10/c10_1_german_tank.py`

In [2]:
import numpy as np

rng = np.random.default_rng(42)
theta, n = 100.0, 10
samples = rng.uniform(0.0, theta, size=(200_000, n))
umvue = ((n + 1.0) / n) * np.max(samples, axis=1)
print(f"UMVUE variance: {np.var(umvue, ddof=1):.4f} (theory: {theta**2/(n*(n+2)):.4f})")
mm = 2 * samples.mean(axis=1)
print(f"MM (2*Xbar) variance: {np.var(mm, ddof=1):.4f} (theory: {theta**2/(3*n):.4f})")

UMVUE variance: 83.0691 (theory: 83.3333)
MM (2*Xbar) variance: 332.5476 (theory: 333.3333)


## The German Tank Problem (continued)

Discrete version: suppose $k=8$ tanks have been captured without replacement from $N=250$ serial numbers.

*Source file:* `code/ch10/c10_2_german_tank_discrete.py`

In [3]:
import numpy as np

rng = np.random.default_rng(42)
N, k, R = 250, 8, 200_000
m = np.array([rng.choice(N, k, replace=False).max() + 1 for _ in range(R)])
umvue = m * (1 + 1/k) - 1
print(f"m(1+1/k)-1 : mean={umvue.mean():.2f} (target {N}) | std={umvue.std():.2f}")
print(f"2*mean - 1 : ", end="")
o = np.array([rng.choice(N, k, replace=False).mean() + 1 for _ in range(R)])
print(f"mean={(2*o - 1).mean():.2f} | std={(2*o - 1).std():.2f}")
print(f"Theory std(UMVUE) = {np.sqrt((N - k)*(N + 1)/(k*(k + 2))):.2f}")

m(1+1/k)-1 : mean=249.91 (target 250) | std=27.65
2*mean - 1 : 

mean=250.12 | std=50.36
Theory std(UMVUE) = 27.55


## Basu's Theorem: $\bar X\indep S^2$ Only for the Normal

*Source file:* `code/ch10/c10_3_basu_only_normal.py`

In [4]:
import numpy as np

rng = np.random.default_rng(42)
n, R = 10, 300_000
dists = {"Normal": lambda: rng.normal(0, 1, (R, n)),
              "Expon. ": lambda: rng.exponential(1, (R, n)),
              "Uniform": lambda: rng.uniform(0, 1, (R, n))}
for name, gen in dists.items():
    X = gen()
    xb, s2 = X.mean(axis=1), X.var(axis=1, ddof=1)
    print(f"{name:7s} | Corr(Xbar, S^2) = {np.corrcoef(xb, s2)[0,1]:+.4f} | Corr(|Xbar-med|, S^2) = "
          f"{np.corrcoef(np.abs(xb - np.median(xb)), s2)[0,1]:+.4f}")

Normal  | Corr(Xbar, S^2) = +0.0042 | Corr(|Xbar-med|, S^2) = +0.0016


Expon.  | Corr(Xbar, S^2) = +0.6962 | Corr(|Xbar-med|, S^2) = +0.3975
Uniform | Corr(Xbar, S^2) = -0.0000 | Corr(|Xbar-med|, S^2) = -0.2594


## Checking the Characterisation of the UMVUE

In the Bernoulli model, $U=X_1-X_2$ is an unbiased estimator of zero. By Theorem 10.9, the UMVUE $\bar X$ must be uncorrelated with $U$, whereas the non-UMVUE $W=X_1$ is not.

*Source file:* `code/ch10/c10_4_umvue_characterisation_check.py`

In [5]:
import numpy as np

rng = np.random.default_rng(42)
p, n, R = 0.3, 6, 1_000_000
X = rng.binomial(1, p, size=(R, n))
U = X[:, 0] - X[:, 1]
T = X.sum(axis=1)
print(f"Cov(Xbar, U) = {np.cov(X.mean(axis=1), U)[0,1]:+.5f}  (UMVUE -> 0)")
print(f"Cov(X_1,  U) = {np.cov(X[:, 0], U)[0,1]:+.5f}  (theory p(1-p) = {p*(1-p):.5f})")
est = T*(n - T)/(n*(n - 1))
print(f"T(n-T)/(n(n-1)) mean = {est.mean():.5f} | target p(1-p) = {p*(1-p):.5f}")
print(f"Xbar(1-Xbar)    mean = {(T/n*(1 - T/n)).mean():.5f} | theory {(n-1)/n*p*(1-p):.5f}")

Cov(Xbar, U) = -0.00017  (UMVUE -> 0)
Cov(X_1,  U) = +0.21002  (theory p(1-p) = 0.21000)
T(n-T)/(n(n-1)) mean = 0.20996 | target p(1-p) = 0.21000
Xbar(1-Xbar)    mean = 0.17497 | theory 0.17500
